<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 16 · SVM và kernel</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Lề cực đại, vector hỗ trợ, lề mềm với C, kernel trick tính tay được, và SVM RBF trên dữ liệu thật.</p>
</div>

### Sau notebook này bạn sẽ

- Tính tay độ rộng lề và nhận ra vector hỗ trợ.
- Đọc C như núm vặn regularization.
- Kiểm kernel trick bằng tay và chỉnh gamma của RBF.
- Chuẩn hóa, chọn C và gamma bằng GridSearchCV.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch16-svm-kernel/01-le-cuc-dai/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Lề cực đại](#trang-1)
2. [Trang 2 · Lề mềm và tham số C](#trang-2)
3. [Trang 3 · Kernel trick](#trang-3)
4. [Trang 4 · SVM trên dữ liệu thật](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Lề cực đại

SVM lề cứng trên tám điểm.

*Đọc kèm trên web: [Trang 1 · Lề cực đại](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch16-svm-kernel/01-le-cuc-dai/)*

In [ ]:
import numpy as np                                     # tính toán
from sklearn.svm import SVC                            # SVM
X = np.array([[1, 1], [2, 1], [1, 2], [2, 3], [4, 4], [5, 4], [4, 5], [5, 6]], dtype=float)   # Bảng 16.1
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])                 # nhãn
m = SVC(kernel="linear", C=1e6).fit(X, y)              # C rất lớn: lề cứng
w, b = m.coef_[0], m.intercept_[0]                     # trọng số, hệ số chặn
print(w.round(4), round(b, 4), m.support_)             # và chỉ số vector hỗ trợ
print(round(2 / np.linalg.norm(w), 4))                 # độ rộng lề (16.1)
print(m.decision_function(X).round(4))                 # w·x + b từng điểm

Output mong đợi:

```text
[0.8 0.4] -3.8 [3 4]
2.2361
[-2.6 -1.8 -2.2 -1.   1.   1.8  1.4  2.6]
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Hai điểm cho đúng -1 và +1: (2; 3) và (4; 4), hai vector hỗ trợ. Lề rộng √5, đúng khoảng cách giữa hai điểm đó.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Lề mềm và tham số C

Một điểm lạc và ba mức C.

*Đọc kèm trên web: [Trang 2 · Lề mềm và tham số C](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch16-svm-kernel/02-le-mem-va-c/)*

In [ ]:
X2 = np.vstack([X, [[3.2, 3.6]]])                    # thêm điểm lạc nhãn 0 sát nhóm xanh
y2 = np.append(y, 0)                                 # nhãn của điểm lạc
for C in [1000, 1, 0.1]:                             # ba mức phạt
    mc = SVC(kernel="linear", C=C).fit(X2, y2)       # lề mềm
    wc = mc.coef_[0]                                 # trọng số
    print(C, wc.round(4), round(mc.intercept_[0], 4), round(2 / np.linalg.norm(wc), 4), round(mc.score(X2, y2), 4))   # w, b, lề, độ chính xác

Output mong đợi:

```text
1000 [2. 1.] -11.0 0.8944 1.0
1 [0.8 0.4] -4.0 2.2361 0.8889
0.1 [0.3533 0.3133] -2.02 4.235 0.8889
```

<div style="border-left:6px solid #1F5F8B;background-color:#E9F1F8;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1F5F8B;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">TRỰC GIÁC</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">C lớn: tin từng điểm, lề co lại còn 0,89 để chia đúng điểm lạc. C nhỏ: tin vào lề rộng, chấp nhận hy sinh điểm lạc.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Kernel trick

Thêm cột x², kernel tính tay, kernel RBF.

*Đọc kèm trên web: [Trang 3 · Kernel trick](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch16-svm-kernel/03-kernel-trick/)*

In [ ]:
x = np.array([-3, -2, -1, 0, 1, 2, 3.0])                                       # bảy điểm trên trục số
y7 = (np.abs(x) <= 1.5).astype(int)                                            # lớp 1 ở giữa
print(SVC(kernel="linear", C=1e6).fit(x[:, None], y7).score(x[:, None], y7))   # chỉ có x: không chia được
P = np.c_[x, x ** 2]                                                            # thêm cột x²
m2 = SVC(kernel="linear", C=1e6).fit(P, y7)                                    # tuyến tính trên (x, x²)
print(m2.score(P, y7), m2.coef_.round(4), m2.intercept_.round(4))              # chia đúng cả 7
a2, b2 = np.array([1.0, 2.0]), np.array([3.0, 1.0])                            # hai điểm
phi = lambda v: np.array([v[0] ** 2, np.sqrt(2) * v[0] * v[1], v[1] ** 2])     # ánh xạ lên 3 chiều
print((a2 @ b2) ** 2, phi(a2) @ phi(b2))                                       # kernel và tích vô hướng tường minh

Output mong đợi:

```text
0.5714285714285714
1.0 [[-0.     -0.6668]] [1.667]
25.0 25.0
```

In [ ]:
from sklearn.datasets import make_moons                                  # hai vầng trăng
from sklearn.model_selection import train_test_split                     # chia
Xm, ym = make_moons(300, noise=0.3, random_state=0)                      # như Chương 14
a, b, c_, e = train_test_split(Xm, ym, test_size=0.3, random_state=0)    # 210 train, 90 test
for g in [0.1, 1, 100]:                                                  # ba giá trị gamma
    mr = SVC(kernel="rbf", gamma=g).fit(a, c_)                           # kernel RBF
    print(g, round(mr.score(a, c_), 4), round(mr.score(b, e), 4))        # train, test

Output mong đợi:

```text
0.1 0.8238 0.8556
1 0.9429 0.9444
100 0.9857 0.8667
```

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI LÀ γ = 1 TỐT NHẤT?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">γ là nghịch đảo bán kính ảnh hưởng của mỗi điểm. γ = 100 thì mỗi điểm chỉ ảnh hưởng chính nó: train 99%, test 87%, giống KNN với k = 1.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · SVM trên dữ liệu thật

Chuẩn hóa, GridSearchCV, xác suất.

*Đọc kèm trên web: [Trang 4 · SVM trên dữ liệu thật](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch16-svm-kernel/04-svm-thuc-te/)*

In [ ]:
from sklearn.datasets import load_breast_cancer                    # ung thư vú
from sklearn.model_selection import GridSearchCV                   # tìm trên lưới
from sklearn.pipeline import make_pipeline                         # ghép bước
from sklearn.preprocessing import StandardScaler                   # chuẩn hóa
dd = load_breast_cancer()                                          # 569 ca
Xb, yb = dd.data, 1 - dd.target                                    # 1 là ác tính
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.25, random_state=0, stratify=yb)   # cách chia Chương 12
print(round(SVC().fit(Xtr, ytr).score(Xte, yte), 4), round(make_pipeline(StandardScaler(), SVC()).fit(Xtr, ytr).score(Xte, yte), 4))   # không và có chuẩn hóa
luoi = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}                     # 16 tổ hợp
gs = GridSearchCV(make_pipeline(StandardScaler(), SVC()), luoi, cv=5).fit(Xtr, ytr)          # 80 lần huấn luyện
print(gs.best_params_, round(gs.best_score_, 4), round(gs.score(Xte, yte), 4))               # tốt nhất, CV, test
mp = make_pipeline(StandardScaler(), SVC(probability=True, random_state=0)).fit(Xtr, ytr)    # bật xác suất
print(mp.predict_proba(Xte[:2]).round(4), mp.decision_function(Xte[:2]).round(4))           # xác suất và khoảng cách có dấu

Output mong đợi:

```text
0.9301 0.951
{'svc__C': 10, 'svc__gamma': 0.01} 0.9836 0.965
[[0.0712 0.9288]
 [0.9955 0.0045]] [ 0.6447 -1.5203]
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Tổ hợp tốt nhất nằm ở mép lưới.</b> Khi đó giá trị thật sự tốt có thể nằm ngoài lưới; mở rộng lưới rồi chạy lại.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Tính tay trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch16-svm-kernel/05-bai-tap/)*

### Bài 16.1 · Điểm mới trong lề

Tính $\mathbf w \cdot \mathbf x + b$ cho điểm $(3; 3)$.
<details><summary>Xem đáp án</summary>

$-0{,}2$: đoán lớp 0 nhưng nằm trong lề, rất gần ranh giới. Code ở ô ngay dưới.
</details>

In [ ]:
print(round(0.8 * 3 + 0.4 * 3 - 3.8, 4))     # w·x + b của điểm (3; 3)

Output mong đợi:

```text
-0.2
```

### Bài 16.2 · Điểm xa nhất

Điểm $(5; 6)$ cách ranh giới bao xa? Xóa nó thì ranh giới đổi không?
<details><summary>Xem đáp án</summary>

$2{,}6/\sqrt{0{,}8} \approx 2{,}907$. Không phải vector hỗ trợ, xóa đi ranh giới không đổi. Code ở ô ngay dưới.
</details>

In [ ]:
print(round(2.6 / np.sqrt(0.8), 4))           # |w·x + b| / ||w||

Output mong đợi:

```text
2.9069
```

### Bài 16.3 · Kernel RBF theo gamma

Tính kernel RBF giữa $(1; 2)$ và $(3; 1)$ với $\gamma = 0{,}1$; 1; 10.
<details><summary>Xem đáp án</summary>

0,607; 0,0067; $1{,}9 \times 10^{-22}$. Cùng một cặp điểm, $\gamma$ quyết định "gần" là bao xa. Code ở ô ngay dưới.
</details>

In [ ]:
print([float(np.exp(-g * np.sum((a2 - b2) ** 2))) for g in [0.1, 1, 10]])   # ba giá trị gamma

Output mong đợi:

```text
[0.6065306597126334, 0.006737946999085467, 1.9287498479639178e-22]
```